# Houses Prices - Modeling and model selection

## Objective

This notebook develops the first modeling stage of the Houses Prices Project.

The goals are:

1.  Establish baseline performance.
2.  Compare multiple regression models.
3.  Select the most promising model using cross-validation.
4.  Improve the selected model through hyperparameter tuning.
5.  Evaluate the final model on the test set.
6.  Interpret results and identify possible failure modes.


## Methodological Principles

This notebook follows a professional machine learning workflow:

* The preprocessing pipeline is reused from the data processing notebook.
*   The test set is kept untouched until the final evaluation.
*   Baseline models are established before tuning.
*   Model selection is based on quantitative metrics.
*   Final conclusions must justify why a model is chosen.


# Models to Evaluate
## Baseline models

* Dummy Regressor
* Linear Regression
* Decision Tree Regression

## Candidate models

*  Random Forest Regression
*  Ridge Regression
*  Lasso Regression
*  Elastic Net Regression
* Support Vector Regression
* HistGradientBoostingRegressor


Main Evaluation Metrics

The main metric for model selection will be Root Mean Squared Error (RMSE).

Additional metrics:

*    Mean Absolute Error (MAE)
 *   Mean Squared Error (MSE)
 *   R2 – Score
 *   Huber Loss



General Structure of a Scikit Learn Model

from sklearn.modelo import Modelo

1. Instanciar (definir hiperparámetros)
model = Modelo(param1=..., param2=...)

2. Entrenar
model.fit(X_train, y_train)

3. Predecir
y_pred = model.predict(X_test)

(Opcional)

probs = model.predict_proba(X_test)


# Main Imports and Configurations

In [1]:
from pathlib import Path
import warnings

import joblib
import numpy as np
import pandas as pd

from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor

from sklearn.metrics import (
    root_mean_squared_error,
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    make_scorer,
)
def huber_loss(y_true, y_pred, delta=1.0):
    error = y_true - y_pred
    is_small_error = np.abs(error) <= delta
    squared_loss = 0.5 * error**2
    linear_loss = delta * (np.abs(error) - 0.5 * delta)
    return np.mean(np.where(is_small_error, squared_loss, linear_loss))
huber_scorer = make_scorer(huber_loss, greater_is_better=False, delta=1.0)
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeRegressor

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
N_SPLITS = 5

CV = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE,
)

SCORING = {
    "neg_root_mean_squared_error": "neg_root_mean_squared_error",
    "neg_mean_absolute_error": "neg_mean_absolute_error",
    "neg_mean_squared_error": "neg_mean_squared_error",
    "r2": "r2",
    "huber_loss": huber_scorer,
}

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 120)

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Setup and Experiment Configuration

This cell imports the models, metrics, and utilities required for the modeling phase.

It also defines:

* a fixed random seed for reproducibility,
 *   the cross-validation strategy,
  *  and the evaluation metrics used throughout the notebook.

Cross-validation is configured with stratified folds in order to preserve the class distribution across splits.

In [3]:
X_train, X_test, y_train, y_test = joblib.load('/content/drive/MyDrive/Split_Data.joblib')

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)
print("Train target mean:", y_train.mean())
print("Test target mean:", y_test.mean())



Train shape: (1168, 29)
Test shape: (292, 29)
Train target mean: 11.50513698630137
Test target mean: 11.506849315068493


# Full Modeling Pipeline

This helper function creates a complete sklearn pipeline by combining:

1.  the preprocessing steps defined in Notebook 2,
2.  the regressor to be trained.

This is a critical design choice because it ensures that:

*   preprocessing is applied consistently,
*  transformations are learned only from training data,
*   and data leakage is avoided during cross-validation and testing.


In [4]:
baseline_models = {
    "dummy": DummyRegressor(strategy="mean"),
    "linear_regression": LinearRegression(
        positive=True,
    ),
    "decision_tree": DecisionTreeRegressor(
        random_state=RANDOM_STATE,
    ),
}

candidate_models = {
    "random_forest": RandomForestRegressor(
        n_estimators=320,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    ),
    "ridge": Ridge(
    ),
    "lasso": Lasso(
    ),
    "elastic_net": ElasticNet(
    ),
    "support_vector": SVR(
    ),
    "hist_gradient_boosting": HistGradientBoostingRegressor(
        random_state=RANDOM_STATE,
    ),
}

# Initial Model Set

This notebook evaluates two groups of models:

## Baseline models

* Dummy Regressor
* Multiple Linear Regression
* Decision Tree Regression

## Candidate models

*  Random Forest Regression
*  Ridge Regression
*  Lasso Regression
*  Elastic Net Regression
* Support Vector Regression
* HistGradientBoostingRegressor

The goal is not to try every possible algorithm, but to compare a small set of relevant models in a structured way.

In [5]:
def evaluate_model_cv(model_name, model, X_train, y_train):
    """
    Evaluate a classification model using stratified cross-validation.

    Parameters
    ----------
    model_name : str
        Name used to identify the model in the output table.
    model : sklearn estimator
        Classification model to evaluate.
    X_train : pandas.DataFrame
        Training feature matrix.
    y_train : pandas.Series
        Training target vector.

    Returns
    -------
    dict
        Dictionary containing the model name and mean cross-validation metrics.
    """
    model_pipeline = model

    cv_results = cross_validate(
        estimator=model_pipeline,
        X=X_train,
        y=y_train,
        cv=CV,
        scoring=SCORING,
        n_jobs=-1,
        return_train_score=False,
    )

    return {
        "model": model_name,
        "cv_RMSE_mean": np.mean(cv_results["test_neg_root_mean_squared_error"]),
        "cv_MAE_mean": np.mean(cv_results["test_neg_mean_absolute_error"]),
        "cv_MSE_mean": np.mean(cv_results["test_neg_mean_squared_error"]),
        "cv_r2_mean": np.mean(cv_results["test_r2"]),
        "cv_huber_loss_mean": np.mean(cv_results["test_huber_loss"]),
    }



Cross-Validation Evaluation Function

This function evaluates a model using stratified cross-validation.

For each model, it:

1.  builds a full pipeline with regressor,
2.  performs cross-validation on the training set,
3.   computes the mean values of the selected evaluation metrics.

This design allows us to compare models under the same conditions before making any tuning or final selection decisions.

In [6]:
baseline_results = []

for model_name, model in baseline_models.items():
    result = evaluate_model_cv(
        model_name=model_name,
        model=model,
        X_train=X_train,
        y_train=y_train,
    )
    baseline_results.append(result)

baseline_results_df = (
    pd.DataFrame(baseline_results)
    .sort_values(by="cv_RMSE_mean", ascending=False)
    .reset_index(drop=True)
)

baseline_results_df

,model,cv_RMSE_mean,cv_MAE_mean,cv_MSE_mean,cv_r2_mean,cv_huber_loss_mean
0,linear_regression,-0.308941,-0.243035,-0.095748,0.664825,-0.047293
1,decision_tree,-0.391783,-0.152412,-0.154129,0.460230,-0.076635
2,dummy,-0.534727,-0.517955,-0.285943,-0.000126,-0.140728


# Baseline Model Results

This table summarizes the performance of the baseline models.

The purpose of this stage is to establish a minimum reference point. In particular, the Dummy Regressor tells us how well a trivial strategy performs, while Linear Regression and Decision Tree provide stronger but still simple baselines.

Any candidate model should clearly outperform these results

In [7]:
candidate_results = []

for model_name, model in candidate_models.items():
    result = evaluate_model_cv(
        model_name=model_name,
        model=model,
        X_train=X_train,
        y_train=y_train,
    )
    candidate_results.append(result)

candidate_results_df = (
    pd.DataFrame(candidate_results)
    .sort_values(by="cv_RMSE_mean", ascending=False)
    .reset_index(drop=True)
)

candidate_results_df

,model,cv_RMSE_mean,cv_MAE_mean,cv_MSE_mean,cv_r2_mean,cv_huber_loss_mean
0,random_forest,-0.279589,-0.149520,-0.078650,0.724652,-0.039039
1,hist_gradient_boosting,-0.293833,-0.173533,-0.086798,0.696113,-0.043078
2,ridge,-0.309869,-0.242245,-0.096299,0.662823,-0.047631
3,elastic_net,-0.373435,-0.294109,-0.139736,0.510753,-0.069161
4,lasso,-0.376444,-0.301461,-0.141971,0.502962,-0.070288
5,support_vector,-0.394451,-0.327844,-0.155885,0.454254,-0.077486


# Candidate Model Results

This table summarizes the performance of the stronger candidate models.
A good amount of these as it stands are as good or worse compared to linear regression, further adjustments of parameters is neccesary.

These models are expected to capture more complex relationships than the baselines. However, they also introduce a greater risk of overfitting, so their performance must be interpreted carefully.

In [8]:
all_results_df = (
    pd.concat([baseline_results_df, candidate_results_df], ignore_index=True)
    .sort_values(by="cv_RMSE_mean", ascending=False)
    .reset_index(drop=True)
)

all_results_df


,model,cv_RMSE_mean,cv_MAE_mean,cv_MSE_mean,cv_r2_mean,cv_huber_loss_mean
0,random_forest,-0.279589,-0.149520,-0.078650,0.724652,-0.039039
1,hist_gradient_boosting,-0.293833,-0.173533,-0.086798,0.696113,-0.043078
2,linear_regression,-0.308941,-0.243035,-0.095748,0.664825,-0.047293
3,ridge,-0.309869,-0.242245,-0.096299,0.662823,-0.047631
4,elastic_net,-0.373435,-0.294109,-0.139736,0.510753,-0.069161
5,lasso,-0.376444,-0.301461,-0.141971,0.502962,-0.070288
6,decision_tree,-0.391783,-0.152412,-0.154129,0.460230,-0.076635
7,support_vector,-0.394451,-0.327844,-0.155885,0.454254,-0.077486
8,dummy,-0.534727,-0.517955,-0.285943,-0.000126,-0.140728


Model Comparison Summary

This consolidated table compares all evaluated models using the same cross-validation setup.

The main criterion for model selection is root mean squared error, while mean absolute error, mean squared error, r2 score and huber loss are used as complementary metrics.

The next step is to select the most promising model or models for hyperparameter tuning.

In [9]:
best_model_name = all_results_df.iloc[0]["model"]
best_model_name

'random_forest'

# Preliminary Best Model Selection

This cell identifies the model with the lowest mean squared error.

This is only a preliminary selection. The final decision will be made after hyperparameter tuning and evaluation on the untouched test set.

# Hyperparameter tuning step

At this stage, we optimize the strongest candidate models using Optuna.

The objective is to improve performance by searching for better hyperparameter configurations while keeping the evaluation rigorous.

Important principles:

  *  Tuning is performed only on the training set.
  *  Cross-validation is used inside the optimization process.
  *  The test set remains untouched until the final evaluation.


In [10]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 7.5 MB/s eta 0:00:00


In [11]:
import optuna

In [12]:
def objective_elastic_net(trial):
    # Define hyperparameters to tune
    model = ElasticNet(
        alpha=trial.suggest_float("alpha", 1e-5, 10.0, log=True),
        l1_ratio=trial.suggest_float("l1_ratio", 0.0, 1.0),
    )

    model_pipeline = model

    cv_results = cross_validate(
        estimator=model_pipeline,
        X=X_train,
        y=y_train,
        cv=CV,
        scoring={"neg_root_mean_squared_error": "neg_root_mean_squared_error"},
        n_jobs=-1,
        return_train_score=False,
    )

    return -np.mean(cv_results["test_neg_root_mean_squared_error"])

# Elastic Net Objective Function

This function defines the search space for Elastic Net and tells Optuna how to evaluate each trial.

For every sampled configuration:

1.    a Elastic Net model is created,
2.  it is wrapped into the full preprocessing pipeline,
3.  cross-validation is performed on the training data,
4.  the RMSE is returned.

This ensures that tuning is based on the same metric used for model comparison.

In [13]:
study_elastic_net = optuna.create_study(direction="minimize")
study_elastic_net.optimize(objective_elastic_net, n_trials=50)

[I 2026-04-15 06:28:31,806] A new study created in memory with name: no-name-77c397b3-d405-4c76-9978-a4bcae832e2a
[I 2026-04-15 06:28:32,133] Trial 0 finished with value: 0.3159041688972149 and parameters: {'alpha': 0.016693583993216906, 'l1_ratio': 0.8345354769856409}. Best is trial 0 with value: 0.3159041688972149.
[I 2026-04-15 06:28:32,498] Trial 1 finished with value: 0.3819121375529124 and parameters: {'alpha': 1.9242815036206953, 'l1_ratio': 0.8017537014625302}. Best is trial 0 with value: 0.3159041688972149.
[I 2026-04-15 06:28:32,816] Trial 2 finished with value: 0.31290887391205835 and parameters: {'alpha': 0.011006357963819572, 'l1_ratio': 0.7018571244023959}. Best is trial 2 with value: 0.31290887391205835.
[I 2026-04-15 06:28:32,966] Trial 3 finished with value: 0.3227784115684782 and parameters: {'alpha': 0.0569541196680574, 'l1_ratio': 0.781049737338206}. Best is trial 2 with value: 0.31290887391205835.
[I 2026-04-15 06:28:33,133] Trial 4 finished with value: 0.310013064

In [14]:
print("Best Elastic Net root mean squared error:", study_elastic_net.best_value)
print("Best Elastic Net params:")
study_elastic_net.best_params

Best Elastic Net root mean squared error: 0.3088190341437903
Best Elastic Net params:


{'alpha': 0.001457796619973083, 'l1_ratio': 0.8404299651354249}

# Elastic Net Tuning Result

This output shows the best cross-validated RMSE found by Optuna for Elastic Net, along with the corresponding hyperparameters.

The next step is to repeat the same process for Ridge, Linear Regression,HistGradientBoosting and Random Forest and compare the tuned models.

In [15]:
def objective_ridge(trial):
    # Define hyperparameters to tune
    model = Ridge(
        alpha=trial.suggest_float("alpha", 1e-5, 1e2, log=True),
        solver=trial.suggest_categorical("solver", ['auto', 'svd', 'cholesky', 'lsqr', 'sag']),
    )

    model_pipeline = model

    cv_results = cross_validate(
        estimator=model_pipeline,
        X=X_train,
        y=y_train,
        cv=CV,
        scoring={"neg_root_mean_squared_error": "neg_root_mean_squared_error"},
        n_jobs=-1,
        return_train_score=False,
    )

    return -np.mean(cv_results["test_neg_root_mean_squared_error"])

# Ridge Objective Function

This function defines the search space for Ridge and evaluates each configuration using cross-validated RMSE.

It's worth noting that Ridge regression is a technique used to address overfitting by adding a penalty to the model's complexity.

In [16]:
study_ridge = optuna.create_study(direction="minimize")
study_ridge.optimize(objective_ridge, n_trials=50)

[I 2026-04-15 06:28:40,754] A new study created in memory with name: no-name-62da80b1-0ada-44c7-8b2a-267d186b7256
[I 2026-04-15 06:28:40,885] Trial 0 finished with value: 0.3106495986777894 and parameters: {'alpha': 0.03479154738357961, 'solver': 'svd'}. Best is trial 0 with value: 0.3106495986777894.
[I 2026-04-15 06:28:44,518] Trial 1 finished with value: 0.31269273657445046 and parameters: {'alpha': 0.0001623888724018204, 'solver': 'sag'}. Best is trial 0 with value: 0.3106495986777894.
[I 2026-04-15 06:28:44,648] Trial 2 finished with value: 0.3103643546980192 and parameters: {'alpha': 0.20680087223829108, 'solver': 'auto'}. Best is trial 2 with value: 0.3103643546980192.
[I 2026-04-15 06:28:44,732] Trial 3 finished with value: 0.30926224433083715 and parameters: {'alpha': 56.01699596591401, 'solver': 'auto'}. Best is trial 3 with value: 0.30926224433083715.
[I 2026-04-15 06:28:44,851] Trial 4 finished with value: 0.31103925085815637 and parameters: {'alpha': 0.07859273644455558, '

In [17]:
print("Best Ridge RMSE:", study_ridge.best_value)
print("Best Ridge params:")
study_ridge.best_params

Best Ridge RMSE: 0.309262098882533
Best Ridge params:


{'alpha': 57.424085345477266, 'solver': 'cholesky'}

# Ridge Tuning Result

This output shows the best cross-validated RMSE found by Optuna for Ridge, along with the corresponding hyperparameters.

In [18]:
def objective_hist_gradient_boosting(trial):
    # Define hyperparameters to tune
    model = HistGradientBoostingRegressor(
        learning_rate=trial.suggest_float("learning_rate", 0.005, 0.3, log=True),
        max_iter=trial.suggest_int("max_iter", 100, 700),
        max_leaf_nodes=trial.suggest_int("max_leaf_nodes", 10, 150),
        max_depth=trial.suggest_int("max_depth", 3, 18),
        min_samples_leaf=trial.suggest_int("min_samples_leaf", 3, 75),
        l2_regularization=trial.suggest_float(
            "l2_regularization", 1e-6, 10.0, log=True
        ),
        random_state=RANDOM_STATE,
    )

    model_pipeline = model

    cv_results = cross_validate(
        estimator=model_pipeline,
        X=X_train,
        y=y_train,
        cv=CV,
        scoring={"neg_root_mean_squared_error": "neg_root_mean_squared_error"},
        n_jobs=-1,
        return_train_score=False,
    )

    return -np.mean(cv_results["test_neg_root_mean_squared_error"])

# HistGradientBoosting Objective Function

This function defines the search space for HistGradientBoosting and evaluates each configuration using cross-validated RMSE.

In [19]:
study_hist_gradient_boosting = optuna.create_study(direction="minimize")
study_hist_gradient_boosting.optimize(objective_hist_gradient_boosting, n_trials=50)

[I 2026-04-15 06:29:10,836] A new study created in memory with name: no-name-ff264090-b014-4e3c-823b-6fc4aa1f4a92
[I 2026-04-15 06:29:23,364] Trial 0 finished with value: 0.2901160595690141 and parameters: {'learning_rate': 0.008236270212736228, 'max_iter': 603, 'max_leaf_nodes': 92, 'max_depth': 13, 'min_samples_leaf': 13, 'l2_regularization': 0.0004258307645261862}. Best is trial 0 with value: 0.2901160595690141.
[I 2026-04-15 06:29:24,905] Trial 1 finished with value: 0.2911330086498417 and parameters: {'learning_rate': 0.09705051122217026, 'max_iter': 305, 'max_leaf_nodes': 25, 'max_depth': 6, 'min_samples_leaf': 60, 'l2_regularization': 0.00014006436725054106}. Best is trial 0 with value: 0.2901160595690141.
[I 2026-04-15 06:29:29,490] Trial 2 finished with value: 0.2921321408057324 and parameters: {'learning_rate': 0.02187093889106647, 'max_iter': 638, 'max_leaf_nodes': 87, 'max_depth': 7, 'min_samples_leaf': 19, 'l2_regularization': 1.417644227396859e-06}. Best is trial 0 with v

In [20]:
print("Best HistGradientBoosting RMSE:", study_hist_gradient_boosting.best_value)
print("Best HistGradientBoosting params:")
study_hist_gradient_boosting.best_params

Best HistGradientBoosting RMSE: 0.280811221123087
Best HistGradientBoosting params:


{'learning_rate': 0.015085102687086571,
 'max_iter': 340,
 'max_leaf_nodes': 150,
 'max_depth': 18,
 'min_samples_leaf': 53,
 'l2_regularization': 9.636193638315381}

# HistGradientBoosting Tuning Result

This output shows the best cross-validated RMSE found by Optuna for HistGradientBoosting, along with the corresponding hyperparameters.

In [21]:
def objective_random_forest(trial):
    # Define hyperparameters to tune
    model = RandomForestRegressor(
        n_estimators=trial.suggest_int("n_estimators", 80, 600),
        max_depth=trial.suggest_int("max_depth", 3, 25),
        min_samples_split=trial.suggest_int("min_samples_split", 2, 25),
        min_samples_leaf=trial.suggest_int("min_samples_leaf", 1, 12),
        max_features=trial.suggest_categorical('max_features', ['sqrt', 'log2', None]),
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )

    model_pipeline = model

    cv_results = cross_validate(
        estimator=model_pipeline,
        X=X_train,
        y=y_train,
        cv=CV,
        scoring={"neg_root_mean_squared_error": "neg_root_mean_squared_error"},
        n_jobs=-1,
        return_train_score=False,
    )

    return -np.mean(cv_results["test_neg_root_mean_squared_error"])

# Random Forest Objective Function

This function defines the search space for Random Forest and evaluates each configuration using cross-validated RMSE.

In [22]:
study_random_forest = optuna.create_study(direction="minimize")
study_random_forest.optimize(objective_random_forest, n_trials=50)

[I 2026-04-15 06:32:01,053] A new study created in memory with name: no-name-e390a9e9-77f1-45a3-a835-c1850b287296
[I 2026-04-15 06:32:09,265] Trial 0 finished with value: 0.2809459893012515 and parameters: {'n_estimators': 356, 'max_depth': 6, 'min_samples_split': 2, 'min_samples_leaf': 10, 'max_features': None}. Best is trial 0 with value: 0.2809459893012515.
[I 2026-04-15 06:32:14,497] Trial 1 finished with value: 0.27754019259788665 and parameters: {'n_estimators': 493, 'max_depth': 18, 'min_samples_split': 18, 'min_samples_leaf': 10, 'max_features': 'log2'}. Best is trial 1 with value: 0.27754019259788665.
[I 2026-04-15 06:32:21,767] Trial 2 finished with value: 0.2800454632736352 and parameters: {'n_estimators': 207, 'max_depth': 12, 'min_samples_split': 16, 'min_samples_leaf': 1, 'max_features': None}. Best is trial 1 with value: 0.27754019259788665.
[I 2026-04-15 06:32:23,362] Trial 3 finished with value: 0.2737444267942809 and parameters: {'n_estimators': 139, 'max_depth': 13, 

In [23]:
print("Best Random Forest RMSE:", study_random_forest.best_value)
print("Best Random Forest params:")
study_random_forest.best_params

Best Random Forest RMSE: 0.2697542403503709
Best Random Forest params:


{'n_estimators': 467,
 'max_depth': 7,
 'min_samples_split': 2,
 'min_samples_leaf': 1,
 'max_features': 'sqrt'}

# Random Forest Tuning Result

This output shows the best cross-validated RMSE found by Optuna for Random Forest, along with the corresponding hyperparameters.

At this point, all tuned candidate models can be compared directly to determine which one should move to final evaluation.

In [24]:
tuned_results_df = pd.DataFrame(
    [
        {
            "model": "ridge_tuned",
            "cv_RMSE_mean": -study_ridge.best_value,
        },
        {
            "model": "elastic_net_tuned",
            "cv_RMSE_mean": -study_elastic_net.best_value,
        },
        {
            "model": "hist_gradient_boosting_tuned",
            "cv_RMSE_mean": -study_hist_gradient_boosting.best_value,
        },
        {
            "model": "random_forest_tuned",
            "cv_RMSE_mean": -study_random_forest.best_value,
        },
    ]
).sort_values(by="cv_RMSE_mean", ascending=False).reset_index(drop=True)

tuned_results_df

,model,cv_RMSE_mean
0,random_forest_tuned,-0.269754
1,hist_gradient_boosting_tuned,-0.280811
2,elastic_net_tuned,-0.308819
3,ridge_tuned,-0.309262


# Tuned Model Comparison

This table compares the best tuned versions of the candidate models.

The model with the closest to zero cross-validated RMSE will be selected for final training and evaluation on the test set.

In [25]:
best_tuned_model_name = tuned_results_df.iloc[0]["model"]
best_tuned_model_name

'random_forest_tuned'

# Final Model Candidate

This cell identifies the best tuned model according to cross-validated RMSE.

This model will now be trained on the full training set and evaluated once on the untouched test set.

In [26]:
final_comparison_df = pd.concat(
    [
        all_results_df[["model", "cv_RMSE_mean"]],
        tuned_results_df.rename(columns={"cv_RMSE_mean": "cv_RMSE_mean"}),
    ],
    ignore_index=True,
).sort_values(by="cv_RMSE_mean", ascending=False).reset_index(drop=True)

final_comparison_df

,model,cv_RMSE_mean
0,random_forest_tuned,-0.269754
1,random_forest,-0.279589
2,hist_gradient_boosting_tuned,-0.280811
3,hist_gradient_boosting,-0.293833
4,elastic_net_tuned,-0.308819
5,linear_regression,-0.308941
6,ridge_tuned,-0.309262
7,ridge,-0.309869
8,elastic_net,-0.373435
9,lasso,-0.376444


# Final Model Comparison

This table compares all evaluated models:

*  baseline models
*   candidate models
*   tuned models

The goal is to select the model with the best cross-validated RMSE, regardless of its complexity.

It is possible that a simpler model outperforms more complex ones, which would be preferable in terms of interpretability and robustness.

# Final Model Selection

The selected model is: random_forest_tuned

Justification:

*  Highest cross-validated RMSE
*  Stability across folds
*  Robustness and less overall overfitting


In [27]:
selected_model_name = final_comparison_df.iloc[0]["model"]
selected_model_name

'random_forest_tuned'

In [28]:
if selected_model_name == "dummy":
    final_estimator = DummyRegressor(strategy="mean")

elif selected_model_name == "linear_regression":
    final_estimator = LinearRegression(
        positive=True,
    )

elif selected_model_name == "decision_tree":
    final_estimator = DecisionTreeRegressor(
        random_state=RANDOM_STATE,
    )

elif selected_model_name == "random_forest":
    final_estimator = RandomForestRegressor(
        n_estimators=320,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )

elif selected_model_name == "hist_gradient_boosting":
    final_estimator = HistGradientBoostingRegressor(
        random_state=RANDOM_STATE,
    )

elif selected_model_name == "random_forest_tuned":
    final_estimator = RandomForestRegressor(
        **study_random_forest.best_params,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )

elif selected_model_name == "hist_gradient_boosting_tuned":
    final_estimator = HistGradientBoostingRegressor(
        **study_hist_gradient_boosting.best_params,
        random_state=RANDOM_STATE,
    )

elif selected_model_name == "support_vector":
    final_estimator = SVR(
    )

elif selected_model_name == "lasso":
    final_estimator = Lasso(
    )

elif selected_model_name == "elastic_net":
    final_estimator = ElasticNet(
    )

elif selected_model_name == "ridge":
    final_estimator = Ridge(
    )

elif selected_model_name == "ridge_tuned":
    final_estimator = Ridge(
        **study_ridge.best_params,
    )

elif selected_model_name == "linear_regression":
    final_estimator = LinearRegression(
        positive=True,
    )

elif selected_model_name == "elastic_net_tuned":
    final_estimator = ElasticNet(
        **study_elastic_net.best_params,
    )

else:
    raise ValueError(f"Unknown selected model: {selected_model_name}")

In [29]:
def make_model_pipeline(model):
    """
    Build a full modeling pipeline that combines preprocessing
    and classification in a single reproducible object.

    Parameters
    ----------
    preprocess_pipeline : sklearn transformer
        Preprocessing pipeline defined in Notebook 1.
    model : sklearn estimator
        Classification model.

    Returns
    -------
    sklearn.pipeline.Pipeline
        Full pipeline with preprocessing and model.
    """
    return Pipeline(
        steps=[
            ("model", model),
        ]
    )

In [30]:
final_model_pipeline = make_model_pipeline(
    model=final_estimator,
)

final_model_pipeline.fit(X_train, y_train)

Pipeline(steps=[('model',
                 RandomForestRegressor(max_depth=7, max_features='sqrt',
                                       n_estimators=467, n_jobs=-1,
                                       random_state=42))])

# Final Training and Final Test Performance

This cell trains the selected model using the full training set.

At this point:

 *   all model selection decisions have already been made,
 *   hyperparameter tuning has already been completed if necessary,
 *   and the model is now ready for a single final evaluation on the test set.


In [31]:
y_pred = final_model_pipeline.predict(X_test)

if hasattr(final_model_pipeline, "predict_proba"):
    y_proba = final_model_pipeline.predict_proba(X_test)[:, 1]
else:
    y_proba = None

Now we can see what the model metrics (How the model performs) are using the test set.

In [32]:
test_results = {
    "MAE": mean_absolute_error(y_test, y_pred),
    "RMSE": root_mean_squared_error(y_test, y_pred),
    "MSE": mean_squared_error(y_test, y_pred),
    "r2_score": r2_score(y_test, y_pred),
    "huber_loss": huber_loss(y_test, y_pred),
}

if y_proba is not None:
    test_results["r2_score"] = r2_score(y_test, y_proba)

test_results_df = pd.DataFrame([test_results])
test_results_df

,MAE,RMSE,MSE,r2_score,huber_loss
0,0.172594,0.282891,0.080027,0.718412,0.040014


In [35]:
importances = final_estimator.feature_importances_
feature_importance_df = pd.DataFrame({'feature': X_train.columns, 'importance': importances})
feature_importance_df.sort_values('importance', ascending=False)

,feature,importance
6,Log_TotalSF,0.167648
7,Log_GrLivArea,0.122300
0,OverallQual,0.100553
17,YearBuilt,0.088146
9,TotalBath,0.059161
1,ExterQual,0.046914
12,GarageCars,0.044203
3,BsmtQual,0.043851
8,Log_1stFlrSF,0.039125
4,GarageFinish,0.039053


Here we can see the selected model feature importance.

#Detailed Evaluation

All metrics used provide a more complete view of model behavior than a single metric.

In particular, they help in:

 *   Evaluating Regression Goodness-of-Fit,
 *   Evaluating weighted and unweighted error,
 *   Assign fair value to outliers.


# Final Conclusions

This notebook completed the first full modeling cycle of the Houses prices project.
Main outcomes

 *   A baseline performance level was established using simple models.
 *   Multiple regressors were compared under the same cross-validation setup.
 *   The strongest candidate models were tuned with Optuna.
 *   A final model was selected based on cross-validated RMSE.
 *   The selected model was evaluated on the test set.
  *  Error analysis and interpretation were performed.

Key methodological strengths

 *   The test set remained untouched until final evaluation.
 *   Model selection was based on quantitative evidence.
  *  Simpler models were not discarded unnecessarily.

Next steps

 *   refine feature engineering if needed,
 *   analyze subgroup failures more deeply,
  *  consider calibration or threshold adjustment,
  *  and prepare the model pipeline for deployment.
